In [1]:
# Setup
import os, glob, json, subprocess
import numpy as np, pandas as pd
subprocess.run(["pip","-q","install","biopython","freesasa"], check=True)

from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

PROJ = "/content/drive/MyDrive/il2_mimetic"
assert os.path.isfile(f"{PROJ}/stage1_motif/params.json"), "stale mount?"
OUT4 = f"{PROJ}/stage4_fold"
PRED = f"{PROJ}/stage5_complex/preds"
ADD  = f"{PROJ}/analysis_additional"; os.makedirs(ADD, exist_ok=True)

DESIGNS = ["pad0_len100_37_fixContacts_2","pad0_len100_2_fixContacts_3",
           "pad0_len100_2_fixContacts_0","pad0_len100_13_fixContacts_3",
           "pad0_len100_15_fixAll_4","pad0_len100_22_fixAll_0",
           "pad2_len100_7_fixAll_3"]

CHAIN = json.load(open(f"{PROJ}/stage1_motif/params.json"))["chains"]
REFS = {"native IL-2": (f"{PROJ}/2B5I.pdb", CHAIN["IL2"]),
        "Neo-2/15":    (f"{PROJ}/stage3_seqs/benchmark/6DG6.pdb", "A")}

def design_pdb(d):      # ESMFold monomer model
    return f"{OUT4}/pdb/{d}.pdb"

print(f"{len(DESIGNS)} designs, {len(REFS)} references")
for d in DESIGNS:
    assert os.path.exists(design_pdb(d)), d
print("all model files present")

Mounted at /content/drive
7 designs, 2 references
all model files present


In [5]:
# Geometric plausibility: Ramachandran angles and steric clashes
#
# Three corrections were needed before this gave meaningful numbers. Each was
# caught by a reference structure behaving implausibly, not by inspecting the
# designs.
#
# 1. Cys-Cys in native IL-2 was flagged as the worst "clash" in the panel.
#      It is a disulfide bond: SG-SG sits at ~2.05 A, far inside the sum of
#      sulphur van der Waals radii (3.60 A). Covalent bonds are not clashes.
#      -> SG-SG pairs excluded.
#
# 2. Several IL-2 contacts were against HOH residues. Crystal structures
#      carry ordered water; predicted models do not, so including solvent made
#      the reference look worse than the designs for no structural reason.
#      -> HETATM records (waters, ligands, sugars) excluded.
#
# 3. Every remaining reference "clash" was an O...OG or OD2...ND2 pair at
#      2.4-2.6 A. Those are hydrogen bonds. Polar atoms legitimately approach
#      closer than their van der Waals sum, implying an H-bond is.
#      -> N/O pairs at >= 2.4 A counted as H-bonds, not clashes. Below 2.4 A
#         they remain flagged: a salt bridge sits at 2.7-3.0 A, so anything
#         shorter is a genuine geometry error.
#
# Additional caveats:
# - No hydrogens in these models, so clash detection is heavy-atom only.
# - Ramachandran classification uses coarse boxes, not real contour maps.
# - Models are unminimised ESMFold output. Local contacts at this scale are a
#   known artefact of structure prediction and would largely relax away under
#   FastRelax or an OpenMM minimisation, which was not performed.
# - Valid as a comparison between structures processed identically; not an
#   absolute quality score.

import numpy as np, pandas as pd
from Bio.PDB import PDBParser, PPBuilder
from scipy.spatial import cKDTree

parser, ppb = PDBParser(QUIET=True), PPBuilder()

VDW = {"C": 1.70, "N": 1.55, "O": 1.52, "S": 1.80}   # heavy atoms only
AA3 = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET "
          "PHE PRO SER THR TRP TYR VAL".split())
HBOND_MIN = 2.4      # N/O pairs at or above this are H-bonds, not clashes
OVERLAP   = 0.4      # van der Waals overlap (A) above which a contact counts


def rama_class(phi, psi, resname):
    """Coarse Ramachandran region. Glycine is reported separately because its
    allowed region is far wider and it would otherwise dominate the outliers."""
    if phi is None or psi is None:
        return "terminal"
    if resname == "GLY":
        return "glycine"
    p, s = np.degrees(phi), np.degrees(psi)
    if -160 <= p <= -20 and -120 <= s <= 50:          return "alpha"
    if -180 <= p <= -40 and (s >= 90 or s <= -150):   return "beta"
    if 20 <= p <= 100 and -20 <= s <= 90:             return "L-alpha"
    return "outlier"


def tier1_geometry(path, chain_id=None):
    structure = parser.get_structure("x", path)
    chain = structure[0][chain_id] if chain_id else next(iter(structure[0]))

    # ---- backbone dihedrals ----
    classified = []
    for peptide in ppb.build_peptides(chain):
        for residue, (phi, psi) in zip(peptide, peptide.get_phi_psi_list()):
            classified.append((residue.get_resname(), residue.id[1],
                               rama_class(phi, psi, residue.get_resname())))
    scored   = [c for c in classified if c[2] != "terminal"]
    outliers = [f"{name}{num}" for name, num, cls in scored if cls == "outlier"]

    # ---- atom selection: standard residues only, no waters or ligands (fix 2) ----
    atoms = [a for a in chain.get_atoms()
             if a.element in VDW
             and a.get_parent().get_resname() in AA3
             and a.get_parent().id[0] == " "]

    xyz     = np.array([a.coord for a in atoms])
    radii   = np.array([VDW[a.element] for a in atoms])
    resnum  = np.array([a.get_parent().id[1] for a in atoms])
    element = [a.element for a in atoms]
    atname  = [a.get_id() for a in atoms]
    resname = [a.get_parent().get_resname() for a in atoms]

    clashes, n_hbonds = [], 0
    for i, j in cKDTree(xyz).query_pairs(r=4.0):
        if abs(resnum[i] - resnum[j]) < 2:            # adjacent residues are bonded
            continue
        if atname[i] == "SG" and atname[j] == "SG":   # disulfide, not a clash (fix 1)
            continue

        dist    = float(np.linalg.norm(xyz[i] - xyz[j]))
        overlap = (radii[i] + radii[j]) - dist
        if overlap <= OVERLAP:
            continue

        both_polar = element[i] in "NO" and element[j] in "NO"
        if both_polar and dist >= HBOND_MIN:          # hydrogen bond (fix 3)
            n_hbonds += 1
            continue

        clashes.append((f"{resname[i]}{resnum[i]}:{atname[i]}",
                        f"{resname[j]}{resnum[j]}:{atname[j]}",
                        round(overlap, 2), round(dist, 2),
                        "polar<2.4A" if both_polar else "steric"))
    clashes.sort(key=lambda c: -c[2])

    n = len(scored)
    return dict(
        n_res          = n,
        pct_alpha      = round(100 * sum(c[2] == "alpha" for c in scored) / n, 1),
        pct_beta       = round(100 * sum(c[2] == "beta"  for c in scored) / n, 1),
        n_outliers     = len(outliers),
        pct_outliers   = round(100 * len(outliers) / n, 2),
        outliers       = ";".join(outliers[:6]),
        n_hbonds       = n_hbonds,
        n_clashes      = len(clashes),
        worst_overlap  = clashes[0][2] if clashes else 0.0,
        worst_detail   = " | ".join(f"{a}-{b} d{d} [{t}]" for a, b, o, d, t in clashes[:3]),
    )


results = {name: tier1_geometry(path, chain) for name, (path, chain) in REFS.items()}
results.update({d: tier1_geometry(design_pdb(d)) for d in DESIGNS})
tier1 = pd.DataFrame(results).T

pd.set_option("display.width", 240)
print(tier1[["n_res","pct_alpha","pct_beta","n_outliers","pct_outliers",
             "n_hbonds","n_clashes","worst_overlap"]].to_string())
print("\nworst contacts:")
for name, row in tier1.iterrows():
    print(f"  {name:30s} {row.worst_detail or '(none)'}")
print("\nRamachandran outliers:")
for name, row in tier1.iterrows():
    if row.outliers:
        print(f"  {name:30s} {row.outliers}")

tier1.to_csv(f"{ADD}/tier1_geometry.tsv", sep="\t")

                             n_res pct_alpha pct_beta n_outliers pct_outliers n_hbonds n_clashes worst_overlap
native IL-2                    114      77.2     17.5          4         3.51        5         2          0.45
Neo-2/15                        98      92.9      4.1          2         2.04        5         0           0.0
pad0_len100_37_fixContacts_2   109      93.6      4.6          1         0.92        2         6          0.59
pad0_len100_2_fixContacts_3    113      93.8      4.4          0          0.0        3         7          1.24
pad0_len100_2_fixContacts_0    113      93.8      4.4          0          0.0        1        10          1.02
pad0_len100_13_fixContacts_3    96      92.7      5.2          0          0.0        2         8          1.06
pad0_len100_15_fixAll_4        103      94.2      4.9          0          0.0        1         3          0.86
pad0_len100_22_fixAll_0        105      94.3      2.9          3         2.86        4         4          0.93
p

In [7]:
# Fold stability: buried polar criteron and hydrophobic core
#
# Two corrections, again driven by controls rather than by the designs.
#
# 1. The first version counted every polar atom, backbone included. All these
#      designs are 100% alpha-helical, and a helix satisfies its own backbone
#      N and O by the i -> i+4 hydrogen bond. That is ~80% of the polar atoms,
#      guaranteed to pass, and it diluted the signal to nothing: the scrambled
#      control scored the same as native IL-2.
#      -> restricted to SIDECHAIN polar atoms, which are what sequence design
#         actually chose.
#
# 2. A 3.2 A hydrogen-bond cutoff flagged partners sitting at 3.26 and 3.39 A
#      - both perfectly good H-bonds that failed by a hair. The conventional
#      heavy-atom cutoff is 3.5 A.
#      -> 3.5 A used as primary, 3.2 A reported as a sensitivity check.
#
# Additional caveats:
#   - Distance only. No hydrogens are placed and no donor-acceptor angles are
#     checked, so this over-calls relative to Rosetta's implementation.
#   - No ordered water. Real buried polars are often satisfied by a trapped
#     water molecule, which none of these models contain. Native IL-2 scoring
#     3 by this measure shows the absolute scale is not calibrated to "defect".
#   - Core packing here is burial statistics, not void detection. No cavity
#     volume, no packstat, no dG unfolding - those need a force field.

import os, numpy as np, pandas as pd, freesasa
from scipy.spatial import cKDTree
freesasa.setVerbosity(freesasa.silent)

TMP = "/content/_tier2"; os.makedirs(TMP, exist_ok=True)

POLAR       = set("NO")
BACKBONE    = {"N", "O", "OXT"}
HYDROPHOBIC = set("ALA VAL LEU ILE MET PHE TRP PRO CYS".split())

# maximum solvent-accessible area per residue (Tien et al. 2013, theoretical),
# used to convert absolute SASA into a relative burial fraction
MAXASA = {"ALA":129,"ARG":274,"ASN":195,"ASP":193,"CYS":167,"GLN":225,"GLU":223,
          "GLY":104,"HIS":224,"ILE":197,"LEU":201,"LYS":236,"MET":224,"PHE":240,
          "PRO":159,"SER":155,"THR":172,"TRP":285,"TYR":263,"VAL":174}

BURIED_ATOM = 1.0     # atom SASA (A^2) at or below which it counts as buried
BURIED_RES  = 0.10    # relative residue SASA below which the residue counts as buried


def _write_clean_pdb(path, chain_id, dest):
    """Standard residues only; no waters, ligands or sugars. Returns the lines
    written, keyed for coordinate lookup."""
    keep, coords = [], {}
    for l in open(path):
        if not l.startswith("ATOM"): continue
        if l[17:20].strip() not in AA3: continue
        if chain_id and l[21] != chain_id: continue
        alt = l[16]
        if alt not in (" ", "A"): continue          # keep one conformer only
        keep.append(l)
        key = (l[21], l[22:27].strip(), l[12:16].strip())
        coords[key] = (np.array([float(l[30+8*i:38+8*i]) for i in range(3)]),
                       (l[76:78].strip() or l[12:16].strip().lstrip("0123456789")[0]))
    with open(dest, "w") as fh:
        fh.writelines(keep); fh.write("END\n")
    return coords


def tier2_stability(path, chain_id=None, hb_cut=3.5):
    tmp = f"{TMP}/t2.pdb"
    coord_lookup = _write_clean_pdb(path, chain_id, tmp)

    st  = freesasa.Structure(tmp)
    res = freesasa.calc(st)

    # ---- build everything from freesasa's atom list (fix 3) ----
    xyz, area, element, atname, resname, resnum = [], [], [], [], [], []
    for i in range(st.nAtoms()):
        an = st.atomName(i).strip()
        rn = st.residueNumber(i).strip()
        ch = st.chainLabel(i).strip() or " "
        hit = coord_lookup.get((ch, rn, an))
        if hit is None:
            continue
        xyz.append(hit[0]); element.append(hit[1])
        area.append(res.atomArea(i))
        atname.append(an); resname.append(st.residueName(i).strip()); resnum.append(int(rn))

    xyz  = np.array(xyz); area = np.array(area)
    assert len(xyz) == len(area) == len(atname), "atom arrays out of step"

    # ---- buried unsatisfied SIDECHAIN polars (fixes 1 and 2) ----
    polar_idx = [i for i, e in enumerate(element) if e in POLAR]
    tree = cKDTree(xyz[polar_idx])
    unsatisfied, n_buried_sc = [], 0
    for i in [k for k in polar_idx if atname[k] not in BACKBONE]:
        if area[i] > BURIED_ATOM:
            continue
        n_buried_sc += 1
        near = tree.query_ball_point(xyz[i], r=hb_cut)
        partners = [polar_idx[k] for k in near
                    if polar_idx[k] != i and resnum[polar_idx[k]] != resnum[i]]
        if not partners:
            unsatisfied.append(f"{resname[i]}{resnum[i]}:{atname[i]}")

    # ---- hydrophobic core composition ----
    res_area, res_name = {}, {}
    for i, rn in enumerate(resnum):
        res_area[rn] = res_area.get(rn, 0.0) + area[i]
        res_name[rn] = resname[i]
    rel = {rn: res_area[rn] / MAXASA.get(res_name[rn], 200) for rn in res_area}

    buried   = [rn for rn, r in rel.items() if r < BURIED_RES]
    hydroph  = [rn for rn in res_area if res_name[rn] in HYDROPHOBIC]
    buried_h = [rn for rn in buried if res_name[rn] in HYDROPHOBIC]

    return dict(
        n_res              = len(res_area),
        n_buried_sc_polar  = n_buried_sc,
        buns               = len(unsatisfied),
        buns_per_100res    = round(100 * len(unsatisfied) / len(res_area), 2),
        buns_where         = ";".join(unsatisfied[:6]),
        n_buried_res       = len(buried),
        pct_buried         = round(100 * len(buried) / len(res_area), 1),
        pct_core_hydroph   = round(100 * len(buried_h) / len(buried), 1) if buried else np.nan,
        pct_hydroph_buried = round(100 * len(buried_h) / len(hydroph), 1) if hydroph else np.nan,
    )


rows = {name: tier2_stability(p, c) for name, (p, c) in REFS.items()}
rows.update({d: tier2_stability(design_pdb(d)) for d in DESIGNS})
tier2 = pd.DataFrame(rows).T

strict = {name: tier2_stability(p, c, hb_cut=3.2)["buns"] for name, (p, c) in REFS.items()}
strict.update({d: tier2_stability(design_pdb(d), hb_cut=3.2)["buns"] for d in DESIGNS})
tier2["buns_at_3.2A"] = pd.Series(strict)

pd.set_option("display.width", 240)
print(tier2[["n_res","n_buried_sc_polar","buns","buns_at_3.2A","buns_per_100res",
             "n_buried_res","pct_buried","pct_core_hydroph",
             "pct_hydroph_buried"]].to_string())
print("\nunsatisfied buried sidechain polars:")
for name, row in tier2.iterrows():
    if row.buns_where:
        print(f"  {name:30s} {row.buns_where}")

tier2.to_csv(f"{ADD}/tier2_stability.tsv", sep="\t")

                             n_res n_buried_sc_polar buns  buns_at_3.2A buns_per_100res n_buried_res pct_buried pct_core_hydroph pct_hydroph_buried
native IL-2                    120                 6    2             2            1.67           33       27.5             90.9               53.6
Neo-2/15                       100                 3    0             1             0.0           27       27.0             92.6               56.8
pad0_len100_37_fixContacts_2   111                 4    1             1             0.9           28       25.2             71.4               50.0
pad0_len100_2_fixContacts_3    115                10    3             6            2.61           26       22.6             69.2               48.6
pad0_len100_2_fixContacts_0    115                 5    1             3            0.87           24       20.9             70.8               43.6
pad0_len100_13_fixContacts_3    98                 7    1             2            1.02           23       23.5 

In [8]:
# Interface analysis: which residues actually carry the binding
#
# The project's central claim is that the grafted motif presents the receptor
# surface. That has been asserted from the design protocol, never measured.
# This computes per-residue buried surface area on the design chain and splits
# it by whether the position was one RFdiffusion held fixed.
#
# Implementation note: SASA is summed directly from freesasa's own atom list by
# (chain, residue number). Fold stability analysis hit an IndexError from zipping
# freesasa's atoms against raw ATOM lines, where freesasa silently drops alternate conformers
# and nonstandard records, so the two lists differ in length. Working only in
# freesasa's indexing avoids the problem rather than patching around it.
#
# Additional caveats:
#   - Computed on Boltz-2 complex predictions, not experimental structures.
#   - Buried surface area is geometry, not energy. A residue burying a lot of
#     area is not necessarily an energetic hotspot; that needs alanine scanning
#     with a force field, which is out of scope here.
#   - Single seed. The alpha interface was shown to move substantially between
#     seeds, so alpha-side numbers here are indicative only.

import os, glob, numpy as np, pandas as pd, freesasa
freesasa.setVerbosity(freesasa.silent)
TMP = "/content/_tier3"; os.makedirs(TMP, exist_ok=True)


def _chain_lines(path, chains):
    return [l for l in open(path)
            if l.startswith("ATOM")
            and l[17:20].strip() in AA3
            and l[21] in chains
            and l[16] in (" ", "A")]


def _sasa_by_residue(lines, tag, chain_of_interest="A"):
    """Residue-level SASA for one chain, read straight out of freesasa."""
    p = f"{TMP}/{tag}.pdb"
    with open(p, "w") as fh:
        fh.writelines(lines); fh.write("END\n")
    st  = freesasa.Structure(p)
    res = freesasa.calc(st)
    out = {}
    for i in range(st.nAtoms()):
        if st.chainLabel(i).strip() != chain_of_interest:
            continue
        rn = int(st.residueNumber(i))
        out[rn] = out.get(rn, 0.0) + res.atomArea(i)
    return out


def _motif_positions(design):
    """Positions RFdiffusion held fixed, from the Stage 2 .trb (ground truth,
    rather than inferred from sequence identity - the fixContacts arm
    redesigns 15 of the 36, so identity runs do not recover them)."""
    backbone = "_".join(design.split("_")[:3])     # pad0_len100_37
    variant  = "_".join(design.split("_")[:2])     # pad0_len100
    f = glob.glob(f"{PROJ}/stage2_backbones/{variant}/{backbone}.trb")
    if not f:
        return None
    d = np.load(f[0], allow_pickle=True)
    if hasattr(d, "files"):
        d = {k: d[k] for k in d.files}
    return set((np.array(d["con_hal_idx0"]).ravel() + 1).tolist())


def _interface(path, partner_chains, tag):
    """dSASA per design residue on burial by the given partner chains."""
    A       = _chain_lines(path, {"A"})
    partner = _chain_lines(path, partner_chains)
    if not A or not partner:
        return None
    free  = _sasa_by_residue(A,           f"{tag}_free")
    bound = _sasa_by_residue(A + partner, f"{tag}_bound")
    return {rn: free[rn] - bound.get(rn, 0.0) for rn in free}


def tier3_interface(design, min_dsasa=1.0):
    motif = _motif_positions(design)
    if motif is None:
        return None

    tern = glob.glob(f"{PRED}/boltz_results_{design}_ternary/predictions/"
                     f"{design}_ternary/*.pdb")
    alph = glob.glob(f"{PRED}/boltz_results_{design}_alpha/predictions/"
                     f"{design}_alpha/*.pdb")
    if not tern:
        return None

    d_rec = _interface(tern[0], {"B", "C"}, f"{design}_rec")
    d_alp = _interface(alph[0], {"B"},      f"{design}_alp") if alph else {}

    rec_res = {k: v for k, v in d_rec.items() if v > min_dsasa}
    alp_res = {k: v for k, v in (d_alp or {}).items() if v > min_dsasa}

    total   = sum(rec_res.values())
    from_m  = sum(v for k, v in rec_res.items() if k in motif)
    top     = sorted(rec_res.items(), key=lambda kv: -kv[1])[:8]

    shared  = set(rec_res) & set(alp_res)
    return dict(
        n_motif_pos      = len(motif),
        n_rec_interface  = len(rec_res),
        rec_dsasa        = round(total, 1),
        motif_dsasa      = round(from_m, 1),
        pct_from_motif   = round(100 * from_m / total, 1) if total else np.nan,
        n_alpha_interface= len(alp_res),
        n_shared_face    = len(shared),
        pct_face_shared  = round(100 * len(shared) / len(rec_res), 1) if rec_res else np.nan,
        top_residues     = ";".join(f"{k}{'*' if k in motif else ''}:{v:.0f}" for k, v in top),
    )


rows = {d: tier3_interface(d) for d in DESIGNS}
tier3 = pd.DataFrame({k: v for k, v in rows.items() if v}).T

pd.set_option("display.width", 240)
print(tier3[["n_motif_pos","n_rec_interface","rec_dsasa","motif_dsasa",
             "pct_from_motif","n_alpha_interface","n_shared_face",
             "pct_face_shared"]].to_string())
print("\ntop contributing residues  (* = grafted motif position):")
for name, row in tier3.iterrows():
    print(f"  {name:30s} {row.top_residues}")

tier3.to_csv(f"{ADD}/tier3_interface.tsv", sep="\t")

                             n_motif_pos n_rec_interface rec_dsasa motif_dsasa pct_from_motif n_alpha_interface n_shared_face pct_face_shared
pad0_len100_37_fixContacts_2          36              31    1055.4       959.9           90.9                32            19            61.3
pad0_len100_2_fixContacts_3           36              29    1065.5       991.5           93.1                23            17            58.6
pad0_len100_2_fixContacts_0           36              30    1122.3       958.5           85.4                31            16            53.3
pad0_len100_13_fixContacts_3          36              31    1097.7       981.6           89.4                30            21            67.7
pad0_len100_15_fixAll_4               36              26     956.8       838.9           87.7                36            16            61.5
pad0_len100_22_fixAll_0               36              29    1067.7       942.9           88.3                35            21            72.4
pad2_l

In [9]:
# Do the beta/gammac and alpha epitopes overlap?
#
# Stage 1 established that in native IL-2 the two epitopes share no residues:
# that non-overlap is the premise the whole project rests on. The designs show
# 53-72% shared face, which would mean alpha engages them in a different mode
# than it engages IL-2. This runs the same measurement on the controls so the
# contrast is measured rather than assumed.
#
# No motif positions needed here, so the controls require no .trb file and no
# numbering reconciliation between 2B5I and the Boltz models.

def face_overlap(name, min_dsasa=1.0):
    tern = glob.glob(f"{PRED}/boltz_results_{name}_ternary/predictions/{name}_ternary/*.pdb")
    alph = glob.glob(f"{PRED}/boltz_results_{name}_alpha/predictions/{name}_alpha/*.pdb")
    if not tern or not alph:
        print("  missing predictions:", name); return None

    rec = _interface(tern[0], {"B", "C"}, f"{name}_r")
    alp = _interface(alph[0], {"B"},      f"{name}_a")
    rec_res = {k for k, v in rec.items() if v > min_dsasa}
    alp_res = {k for k, v in alp.items() if v > min_dsasa}
    shared  = rec_res & alp_res

    return dict(
        n_rec   = len(rec_res),
        n_alpha = len(alp_res),
        n_shared= len(shared),
        pct_of_rec_face_shared   = round(100*len(shared)/len(rec_res), 1) if rec_res else np.nan,
        pct_of_alpha_face_shared = round(100*len(shared)/len(alp_res), 1) if alp_res else np.nan,
        rec_residues   = ",".join(map(str, sorted(rec_res))),
        alpha_residues = ",".join(map(str, sorted(alp_res))),
    )

panel = ["ctrl_IL2", "ctrl_Neo", "ctrl_scram"] + DESIGNS
rows  = {n: face_overlap(n) for n in panel}
ov    = pd.DataFrame({k: v for k, v in rows.items() if v}).T

print(ov[["n_rec","n_alpha","n_shared","pct_of_rec_face_shared",
          "pct_of_alpha_face_shared"]].to_string())

print("\nnative IL-2 — are the two epitopes distinct?")
if "ctrl_IL2" in ov.index:
    print("  βγc face:", ov.loc["ctrl_IL2","rec_residues"])
    print("  α face  :", ov.loc["ctrl_IL2","alpha_residues"])

ov.to_csv(f"{ADD}/tier3b_epitope_overlap.tsv", sep="\t")

                             n_rec n_alpha n_shared pct_of_rec_face_shared pct_of_alpha_face_shared
ctrl_IL2                        28      24        0                    0.0                      0.0
ctrl_Neo                        30      24       19                   63.3                     79.2
ctrl_scram                      34      23       21                   61.8                     91.3
pad0_len100_37_fixContacts_2    31      32       19                   61.3                     59.4
pad0_len100_2_fixContacts_3     29      23       17                   58.6                     73.9
pad0_len100_2_fixContacts_0     30      31       16                   53.3                     51.6
pad0_len100_13_fixContacts_3    31      30       21                   67.7                     70.0
pad0_len100_15_fixAll_4         26      36       16                   61.5                     44.4
pad0_len100_22_fixAll_0         29      35       21                   72.4                     60.0


In [10]:
# Developability: exposed hydrophobic surface and aggregation risk
#
# The sequence-level analysis already done (pI, net charge, GRAVY,
# free cysteines, glycosylation sequons, deamidation motifs) and the Boltz
# homodimer panel tested self-association directly. What is missing is the
# structural half: where the hydrophobic surface actually is.
#
# A protein can have unremarkable bulk hydrophobicity and still aggregate if
# that hydrophobicity is concentrated in one exposed patch. Sequence-level
# GRAVY cannot see this without a structure.
#
# Three measures, increasing in specificity:
#   1. total exposed nonpolar area: bulk greasiness of the surface
#   2. largest contiguous patch: the thing that actually drives sticking
#   3. SAP-like score: spatially averaged exposure x hydrophobicity,
#   after Chennamsetty et al. (2009)
#
# Tested hypothesis: pad0_len100_2_fixContacts_0 was the clear
# outlier in the homodimer panel (self ipTM 0.597 against Neo-2/15's 0.336).
# If it also carries the largest exposed hydrophobic patch, the two results
# explain each other.
#
# Additional caveats:
#   - Simplified SAP: per-residue rather than per-atom, and using a normalised
#     Kyte-Doolittle scale rather than Black-Mould. Relative comparison only.
#   - Computed on unminimised monomer models, same limitation as tiers 1-2.
#   - Patch definition (relative SASA > 0.25, Cbeta within 8 A) is a convention,
#     not a standard. Reported alongside the raw exposed-area number, which is
#     threshold-free.

import os, numpy as np, pandas as pd, freesasa
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
freesasa.setVerbosity(freesasa.silent)

TMP = "/content/_tier4"; os.makedirs(TMP, exist_ok=True)

NONPOLAR_EL = set("CS")
HYDROPHOBIC = set("ALA VAL LEU ILE MET PHE TRP PRO CYS TYR".split())

# Kyte-Doolittle, rescaled to 0-1 so the SAP sum is interpretable
KD = {"ILE":4.5,"VAL":4.2,"LEU":3.8,"PHE":2.8,"CYS":2.5,"MET":1.9,"ALA":1.8,
      "GLY":-0.4,"THR":-0.7,"SER":-0.8,"TRP":-0.9,"TYR":-1.3,"PRO":-1.6,
      "HIS":-3.2,"GLU":-3.5,"GLN":-3.5,"ASP":-3.5,"ASN":-3.5,"LYS":-3.9,"ARG":-4.5}
KD01 = {k: (v + 4.5) / 9.0 for k, v in KD.items()}

EXPOSED_REL = 0.25     # relative SASA above which a residue counts as surface-exposed
PATCH_DIST  = 8.0      # Cbeta separation (A) within which exposed residues join a patch


def tier4_surface(path, chain_id=None):
    # reuse the fold stability cleaner: standard residues, one conformer, no solvent
    tmp = f"{TMP}/t4.pdb"
    coord_lookup = _write_clean_pdb(path, chain_id, tmp)

    st  = freesasa.Structure(tmp)
    res = freesasa.calc(st)

    # per-atom, read from freesasa's own indexing
    res_area, res_np_area, res_name, res_cb = {}, {}, {}, {}
    for i in range(st.nAtoms()):
        an = st.atomName(i).strip()
        rn = st.residueNumber(i).strip()
        ch = st.chainLabel(i).strip() or " "
        hit = coord_lookup.get((ch, rn, an))
        if hit is None:
            continue
        xyz, element = hit
        num  = int(rn)
        area = res.atomArea(i)
        res_area[num]    = res_area.get(num, 0.0) + area
        res_name[num]    = st.residueName(i).strip()
        if element in NONPOLAR_EL:
            res_np_area[num] = res_np_area.get(num, 0.0) + area
        if an in ("CB", "CA") and num not in res_cb:
            res_cb[num] = xyz                       # CB, or CA for glycine

    rel = {n: res_area[n] / MAXASA.get(res_name[n], 200) for n in res_area}

    # ---- 1. bulk exposed nonpolar surface ----
    total_sasa = sum(res_area.values())
    total_np   = sum(res_np_area.values())

    # ---- 2. largest contiguous exposed hydrophobic patch ----
    patch_res = [n for n in res_area
                 if rel[n] > EXPOSED_REL and res_name[n] in HYDROPHOBIC and n in res_cb]
    if len(patch_res) > 1:
        pts  = np.array([res_cb[n] for n in patch_res])
        pairs = cKDTree(pts).query_pairs(r=PATCH_DIST)
        if pairs:
            r, c = zip(*pairs)
            adj = coo_matrix((np.ones(len(r)), (r, c)),
                             shape=(len(patch_res), len(patch_res)))
            n_comp, labels = connected_components(adj, directed=False)
            sizes = np.bincount(labels)
            big   = int(sizes.max())
            members = [patch_res[i] for i in np.where(labels == sizes.argmax())[0]]
        else:
            big, members = 1, patch_res[:1]
    else:
        big, members = len(patch_res), patch_res

    patch_area = sum(res_np_area.get(n, 0.0) for n in members)

    # ---- 3. SAP-like score: exposure x hydrophobicity, spatially averaged ----
    nums = [n for n in res_area if n in res_cb]
    pts  = np.array([res_cb[n] for n in nums])
    tree = cKDTree(pts)
    sap  = {}
    for i, n in enumerate(nums):
        near = tree.query_ball_point(pts[i], r=5.0)
        sap[n] = sum(min(rel[nums[j]], 1.0) * KD01.get(res_name[nums[j]], 0.5)
                     for j in near)

    return dict(
        n_res              = len(res_area),
        total_sasa         = round(total_sasa),
        exposed_nonpolar   = round(total_np),
        pct_surface_nonpolar = round(100 * total_np / total_sasa, 1),
        n_exposed_hydroph  = len(patch_res),
        largest_patch_res  = big,
        largest_patch_area = round(patch_area),
        sap_max            = round(max(sap.values()), 2),
        sap_mean           = round(float(np.mean(list(sap.values()))), 2),
        patch_members      = ",".join(map(str, sorted(members)[:12])),
    )


rows = {name: tier4_surface(p, c) for name, (p, c) in REFS.items()}
rows.update({d: tier4_surface(design_pdb(d)) for d in DESIGNS})
tier4 = pd.DataFrame(rows).T

pd.set_option("display.width", 250)
print(tier4[["n_res","total_sasa","exposed_nonpolar","pct_surface_nonpolar",
             "n_exposed_hydroph","largest_patch_res","largest_patch_area",
             "sap_max","sap_mean"]].to_string())
print("\nlargest exposed hydrophobic patch, residues:")
for name, row in tier4.iterrows():
    print(f"  {name:30s} {row.patch_members}")

tier4.to_csv(f"{ADD}/tier4_surface.tsv", sep="\t")

                             n_res total_sasa exposed_nonpolar pct_surface_nonpolar n_exposed_hydroph largest_patch_res largest_patch_area sap_max sap_mean
native IL-2                    120       7105             3991                 56.2                15                 2                180    0.96     0.31
Neo-2/15                       100       5836             3415                 58.5                11                 2                158    0.88     0.26
pad0_len100_37_fixContacts_2   111       7348             3769                 51.3                10                 2                112    0.73     0.26
pad0_len100_2_fixContacts_3    115       7756             4078                 52.6                10                 2                113    0.76     0.28
pad0_len100_2_fixContacts_0    115       7951             4231                 53.2                10                 2                113     0.8     0.26
pad0_len100_13_fixContacts_3    98       6363             3351  

Hypothesis fails: self ipTM 0.597, the clear homodimer outlier, is entirely unremarkable here: 53.2% nonpolar surface, SAP 0.80, patch area 113 Å². Its self-association is not explained by exposed hydrophobic surface.

In [13]:
# What surface drives self-association?
#
# The exposed-hydrophobic-patch metric above does not discriminate (every
# molecule, references included, returns a largest patch of 2 residues) and
# does not explain the homodimer outlier.
# Alternative hypothesis: the designed receptor-binding face is itself the
# self-association surface, since it was built to bind a protein
# and another copy of the design is a protein.
#
# Compares, per molecule, the chain A residues buried at the homodimer
# interface against those buried at the beta/gammac interface.

RES = f"{PROJ}/results"
master = pd.read_csv(f"{RES}/candidates_master.tsv", sep="\t", index_col=0)

def self_vs_receptor_face(name, min_dsasa=1.0):
    homo = glob.glob(f"{PRED}/boltz_results_{name}_homodimer/predictions/"
                     f"{name}_homodimer/*.pdb")
    tern = glob.glob(f"{PRED}/boltz_results_{name}_ternary/predictions/"
                     f"{name}_ternary/*.pdb")
    if not homo or not tern:
        print("  missing:", name); return None
    self_if = _interface(homo[0], {"B"},     f"{name}_self")
    rec_if  = _interface(tern[0], {"B","C"}, f"{name}_recf")
    s = {k for k, v in self_if.items() if v > min_dsasa}
    r = {k for k, v in rec_if.items()  if v > min_dsasa}
    return dict(n_self=len(s), n_rec=len(r), n_shared=len(s & r),
                pct_self_on_rec_face=round(100*len(s & r)/len(s), 1) if s else np.nan)

panel = ["ctrl_IL2", "ctrl_Neo"] + DESIGNS
rows  = {n: self_vs_receptor_face(n) for n in panel}
sf    = pd.DataFrame({k: v for k, v in rows.items() if v}).T
sf["self_iptm"] = master["self_iptm"]
print(sf.to_string())
sf.to_csv(f"{ADD}/tier4b_self_face.tsv", sep="\t")

                              n_self  n_rec  n_shared  pct_self_on_rec_face  self_iptm
ctrl_IL2                        70.0   28.0      14.0                  20.0     0.8694
ctrl_Neo                        25.0   30.0       8.0                  32.0     0.3359
pad0_len100_37_fixContacts_2    23.0   31.0      13.0                  56.5     0.3542
pad0_len100_2_fixContacts_3     26.0   29.0      16.0                  61.5     0.3466
pad0_len100_2_fixContacts_0     31.0   30.0      14.0                  45.2     0.5973
pad0_len100_13_fixContacts_3    27.0   31.0      21.0                  77.8     0.4445
pad0_len100_15_fixAll_4         17.0   26.0      10.0                  58.8     0.2066
pad0_len100_22_fixAll_0         18.0   29.0      13.0                  72.2     0.3093
pad2_len100_7_fixAll_3          17.0   30.0       9.0                  52.9     0.4257


The second hypothesis fails as well.

Self-association mechanism: not resolved. Two hypotheses were tested for the homodimer outlier (`_0`, self ipTM 0.597 against Neo-2/15's 0.336). Exposed hydrophobic surface does not explain it — `_0` is unremarkable on surface nonpolar fraction (53.2%), peak SAP (0.80) and patch area. Reuse of the receptor-binding face does not explain it either: `_0` has the lowest receptor-face overlap of any design (45.2%) alongside the highest self-association score, and the relationship across the panel is flat.

The homodimer panel was run at a single seed. Given that four-seed replication of the α panel showed ipTM swings of up to 0.26 near the decision threshold, `_0`'s value is within plausible seed noise. The flag is therefore reported as provisional rather than disqualifying, and replicating the homodimer panel is listed as outstanding work.

A general pattern does hold. All designs reuse their receptor-binding face for
self-association (45–78% of the self-interface) substantially more than Neo-2/15 (32%), consistent with the known liability that a surface designed to bind a protein will also bind a copy of itself.

In [14]:
# Merge all metrics

import pandas as pd

RES = f"{PROJ}/results"
master = pd.read_csv(f"{RES}/candidates_master.tsv", sep="\t", index_col=0)

t1  = pd.read_csv(f"{ADD}/tier1_geometry.tsv",        sep="\t", index_col=0)
t2  = pd.read_csv(f"{ADD}/tier2_stability.tsv",       sep="\t", index_col=0)
t3  = pd.read_csv(f"{ADD}/tier3_interface.tsv",       sep="\t", index_col=0)
t3b = pd.read_csv(f"{ADD}/tier3b_epitope_overlap.tsv",sep="\t", index_col=0)
t4  = pd.read_csv(f"{ADD}/tier4_surface.tsv",         sep="\t", index_col=0)
t4b = pd.read_csv(f"{ADD}/tier4b_self_face.tsv",      sep="\t", index_col=0)

# the geometry tables name the controls "native IL-2" / "Neo-2/15"
ren = {"native IL-2": "ctrl_IL2", "Neo-2/15": "ctrl_Neo"}
t1, t2, t4 = t1.rename(index=ren), t2.rename(index=ren), t4.rename(index=ren)

add = pd.DataFrame(index=master.index)

# Plausibility
add["rama_outliers"]   = t1["n_outliers"]
add["pct_rama_out"]    = t1["pct_outliers"]
add["n_clashes"]       = t1["n_clashes"]
add["worst_overlap"]   = t1["worst_overlap"]

# Stability
add["buns_3.5A"]       = t2["buns"]
add["buns_3.2A"]       = t2["buns_at_3.2A"]
add["pct_buried"]      = t2["pct_buried"]
add["pct_core_hydroph"]= t2["pct_core_hydroph"]

# Binding
add["pct_iface_from_motif"] = t3["pct_from_motif"]
add["n_rec_iface_res"]      = t3["n_rec_interface"]
add["pct_face_shared"]      = t3b["pct_of_rec_face_shared"]

# Behavior
add["pct_surface_nonpolar"] = t4["pct_surface_nonpolar"]
add["sap_max"]              = t4["sap_max"]
add["pct_self_on_rec_face"] = t4b["pct_self_on_rec_face"]

full = master.join(add)
full.round(3).to_csv(f"{RES}/candidates_master.tsv", sep="\t")

print(f"{full.shape[1]} columns\n")
cols = ["arm","length","plddt_esm","rama_outliers","worst_overlap",
        "buns_3.5A","pct_core_hydroph","pct_iface_from_motif",
        "seeds_passing","self_iptm","pct_surface_nonpolar"]
pd.set_option("display.width", 260)
print(full[cols].to_string())

38 columns

                                      arm  length  plddt_esm  rama_outliers  worst_overlap  buns_3.5A  pct_core_hydroph  pct_iface_from_motif  seeds_passing  self_iptm  pct_surface_nonpolar
ctrl_IL2                          control   120.0        NaN            4.0           0.45        2.0              90.9                   NaN            NaN     0.8694                  56.2
ctrl_Neo                          control   100.0        NaN            2.0           0.00        0.0              92.6                   NaN            NaN     0.3359                  58.5
ctrl_scram                        control     NaN        NaN            NaN            NaN        NaN               NaN                   NaN            NaN        NaN                   NaN
pad0_len100_13_fixContacts_3  fixContacts    98.0      93.25            0.0           1.06        1.0              73.9                  89.4            1.0     0.4445                  52.7
pad0_len100_15_fixAll_4            fix

In [15]:
# Structure figure: the leading candidate with the grafted motif highlighted
#
# Motif positions come from the Stage 2 .trb (con_hal_idx0), which is ground
# truth. Inferring them from sequence identity against IL-2 does not work for
# the fixContacts arm: it redesigns 15 of the 36 constrained positions, so the
# identical stretches are broken into fragments too short to detect.

import subprocess, glob, numpy as np
subprocess.run(["pip","-q","install","py3Dmol"], check=True)
import py3Dmol

WIN = "pad0_len100_37_fixContacts_2"

backbone = "_".join(WIN.split("_")[:3])          # pad0_len100_37
variant  = "_".join(WIN.split("_")[:2])          # pad0_len100
trb = glob.glob(f"{PROJ}/stage2_backbones/{variant}/{backbone}.trb")[0]
d = np.load(trb, allow_pickle=True)
if hasattr(d, "files"): d = {k: d[k] for k in d.files}

motif = sorted((np.array(d["con_hal_idx0"]).ravel() + 1).tolist())
ref   = sorted((np.array(d["con_ref_idx0"]).ravel() + 1).tolist())
print(f"{len(motif)} grafted positions")

# print as contiguous segments, which is how they should read
def segments(xs):
    out, start = [], xs[0]
    for i in range(1, len(xs) + 1):
        if i == len(xs) or xs[i] != xs[i-1] + 1:
            out.append((start, xs[i-1]))
            if i < len(xs): start = xs[i]
    return out
print("design :", segments(motif))
print("IL-2   :", segments(ref))

v = py3Dmol.view(width=900, height=640)
v.addModel(open(f"{OUT4}/pdb/{WIN}.pdb").read(), "pdb")
v.setViewStyle({"style":"outline","color":"black","width":0.03})
v.setStyle({"cartoon":{"color":"#d4d4d4","thickness":0.9}})
v.addStyle({"resi": motif}, {"cartoon":{"color":"#CC79A7","thickness":1.15}})
v.zoomTo(); v.spin("y", 0.5); v.show()
print("\npink = 36 scaffolded positions carrying IL-2's βγc binding geometry "
      "(90.9% of the receptor interface)")
print("grey = designed de novo")

36 grafted positions
design : [(10, 21), (46, 57), (90, 101)]
IL-2   : [(7, 18), (75, 86), (106, 117)]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


pink = 36 scaffolded positions carrying IL-2's βγc binding geometry (90.9% of the receptor interface)
grey = designed de novo


In [16]:
job = f"{WIN}_ternary"
f = glob.glob(f"{PRED}/boltz_results_{job}/predictions/{job}/*.pdb")[0]
v = py3Dmol.view(width=900, height=640)
v.addModel(open(f).read(), "pdb")
v.setViewStyle({"style":"outline","color":"black","width":0.03})
v.setStyle({"chain":"A"}, {"cartoon":{"color":"#d4d4d4"}})
v.addStyle({"chain":"A","resi":motif}, {"cartoon":{"color":"#CC79A7","thickness":1.15}})
for ch, col in [("B","#0072B2"), ("C","#009E73")]:
    v.setStyle({"chain":ch}, {"cartoon":{"color":col,"opacity":0.85}})
    v.addSurface("VDW", {"opacity":0.45,"color":col}, {"chain":ch})
v.zoomTo({"chain":"A"}); v.zoom(0.85); v.spin("y", 0.5); v.show()
print("pink = grafted motif · blue = IL-2Rβ · green = γc")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

pink = grafted motif · blue = IL-2Rβ · green = γc


In [18]:
# Is it ready for the bench?
#
# 5a. Does the design bind the SAME SITE on the receptor that IL-2 binds?
#       Every interface metric so far is computed on the design side. A design
#       could engage IL-2Rbeta confidently at a completely different surface
#       and interface error, buried area and contact counts would all look
#       identical. If this fails, nothing else in the project matters.
#
# 5b. Is the beta-gammac orientation signalling-competent?
#       IL-2 is an agonist: signalling requires the two receptor chains to be
#       held in a specific relative geometry so their intracellular domains can
#       engage JAK kinases. A molecule can bind both chains and hold them
#       wrongly - that is an antagonist, the opposite of the goal. This gate
#       was pre-registered before Stage 5 and never run.
#
# 5c. Does the design bind in the conformation it folds into on its own?
#       Rearranging between the free and bound states is an entropic cost and a
#       known failure mode. Field threshold is roughly 1.5 A. No earlier metric
#       would catch it.
#
# Chains B (IL-2Rbeta, 214 aa) and C (gammac, 199 aa) are identical sequences
# across every ternary job, so residue numbering corresponds one-to-one between
# complexes and no alignment is needed.
#
# Tools: Biopython superposition plus the solvent-accessibility functions
# already defined.
#
# Caveats:
#   - All three checks use ternary complex predictions, which were run at a
#     single seed. Only the alpha panel was replicated (4 seeds), where ipTM
#     varied by up to 0.26 near the decision threshold. The ternary predictions
#     were never tested for the same instability, so 5b in particular - a
#     geometric comparison between two predicted complexes - carries an
#     unquantified seed-dependent error.
#   - The native IL-2 ternary complex is itself a prediction rather than the
#     crystal structure, so 5a and 5b measure agreement with a predicted
#     reference, not with experimental geometry.

import glob
import numpy as np, pandas as pd
from Bio.PDB import PDBParser, Superimposer

parser = PDBParser(QUIET=True)


def _partner_interface(path, design_chain, partner_chain, tag):
    """dSASA per PARTNER residue when buried by the design chain."""
    d = _chain_lines(path, {design_chain})
    p = _chain_lines(path, {partner_chain})
    if not d or not p:
        return {}
    free  = _sasa_by_residue(p,     f"{tag}_f", chain_of_interest=partner_chain)
    bound = _sasa_by_residue(p + d, f"{tag}_b", chain_of_interest=partner_chain)
    return {r: free[r] - bound.get(r, 0.0) for r in free}


def _ca(path, chain_id):
    """Alpha carbons keyed by residue number, standard residues only."""
    st = parser.get_structure("x", path)
    ch = st[0][chain_id]
    return {r.id[1]: r["CA"] for r in ch if r.id[0] == " " and "CA" in r}


def _first_chain(path):
    return sorted({l[21] for l in open(path) if l.startswith("ATOM")})[0]


def ternary_pdb(name):
    f = glob.glob(f"{PRED}/boltz_results_{name}_ternary/predictions/"
                  f"{name}_ternary/*.pdb")
    return f[0] if f else None


NATIVE = ternary_pdb("ctrl_IL2")
assert NATIVE, "native IL-2 ternary prediction not found"

# reference epitopes: the receptor residues native IL-2 buries
nat_beta = {r for r, v in _partner_interface(NATIVE, "A", "B", "nat_b").items() if v > 1}
nat_gc   = {r for r, v in _partner_interface(NATIVE, "A", "C", "nat_c").items() if v > 1}
print(f"native IL-2 buries {len(nat_beta)} IL-2Rβ residues, {len(nat_gc)} γc residues")
print(f"  IL-2Rβ epitope: {sorted(nat_beta)}")
print(f"  γc epitope    : {sorted(nat_gc)}\n")

PANEL = ["ctrl_Neo"] + DESIGNS          # Neo-2/15 as the reference de novo molecule

rows = {}
for name in PANEL:
    p = ternary_pdb(name)
    if not p:
        print("  missing ternary prediction:", name)
        continue

    # ---- 5a. receptor-side epitope overlap ----
    db = {r for r, v in _partner_interface(p, "A", "B", f"{name}_b").items() if v > 1}
    dg = {r for r, v in _partner_interface(p, "A", "C", f"{name}_c").items() if v > 1}
    pct_b = 100 * len(db & nat_beta) / len(nat_beta) if nat_beta else np.nan
    pct_g = 100 * len(dg & nat_gc)   / len(nat_gc)   if nat_gc   else np.nan

    # ---- 5b. beta-gammac relative orientation ----
    # superpose on IL-2Rbeta, then measure how far gammac has moved
    nb, qb = _ca(NATIVE, "B"), _ca(p, "B")
    ng, qg = _ca(NATIVE, "C"), _ca(p, "C")
    shared_b = sorted(set(nb) & set(qb))
    shared_g = sorted(set(ng) & set(qg))

    sup = Superimposer()
    sup.set_atoms([nb[i] for i in shared_b], [qb[i] for i in shared_b])
    beta_fit = sup.rms
    moving = [qg[i] for i in shared_g]          # apply the transform to gammac
    sup.apply(moving)
    gc_rmsd = float(np.sqrt(np.mean(
        [np.sum((ng[i].coord - qg[i].coord) ** 2) for i in shared_g])))

    # ---- 5c. bound versus free conformation of the design chain ----
    if name.startswith("ctrl_"):
        bound_free, n_cmp = np.nan, 0        # controls have no ESMFold monomer model
    else:
        mono = _ca(design_pdb(name), _first_chain(design_pdb(name)))
        comp = _ca(p, "A")
        shared_a = sorted(set(mono) & set(comp))
        sup2 = Superimposer()
        sup2.set_atoms([mono[i] for i in shared_a], [comp[i] for i in shared_a])
        bound_free, n_cmp = sup2.rms, len(shared_a)

    rows[name] = dict(
        n_beta_contacts         = len(db),
        pct_beta_epitope_shared = round(pct_b, 1),
        n_gc_contacts           = len(dg),
        pct_gc_epitope_shared   = round(pct_g, 1),
        beta_superpos_rmsd      = round(beta_fit, 2),
        gc_orientation_rmsd     = round(gc_rmsd, 2),
        n_res_compared          = n_cmp,
        bound_vs_free_rmsd      = round(bound_free, 2) if n_cmp else np.nan,
    )

tier5 = pd.DataFrame(rows).T
pd.set_option("display.width", 250)
print(tier5.to_string())

# ---- pre-registered and conventional thresholds ----
print("\ngates:")
print("  5b  gc_orientation_rmsd < 2.0 A   (pre-registered before Stage 5)")
print("  5c  bound_vs_free_rmsd  < 1.5 A   (field convention)")
if "gc_orientation_rmsd" in tier5:
    tier5["pass_5b"] = tier5.gc_orientation_rmsd < 2.0
    tier5["pass_5c"] = tier5.bound_vs_free_rmsd.lt(1.5).where(
    tier5.bound_vs_free_rmsd.notna())
    print()
    print(tier5[["pct_beta_epitope_shared","pct_gc_epitope_shared",
                 "gc_orientation_rmsd","pass_5b",
                 "bound_vs_free_rmsd","pass_5c"]].to_string())

tier5.to_csv(f"{ADD}/tier5_prewetlab.tsv", sep="\t")

native IL-2 buries 16 IL-2Rβ residues, 13 γc residues
  IL-2Rβ epitope: [15, 41, 42, 65, 66, 69, 70, 71, 73, 74, 75, 133, 134, 136, 138, 188]
  γc epitope    : [69, 70, 92, 94, 95, 126, 127, 172, 174, 175, 176, 177, 178]

                              n_beta_contacts  pct_beta_epitope_shared  n_gc_contacts  pct_gc_epitope_shared  beta_superpos_rmsd  gc_orientation_rmsd  n_res_compared  bound_vs_free_rmsd
ctrl_Neo                                 20.0                     93.8           16.0                  100.0                1.89                 1.50             0.0                 NaN
pad0_len100_37_fixContacts_2             18.0                     87.5           10.0                   76.9                1.40                 2.07           111.0                0.53
pad0_len100_2_fixContacts_3              16.0                     87.5           12.0                   84.6                1.90                 1.48           115.0                0.37
pad0_len100_2_fixContacts_0       